# KV-cache eviction and a planted complaint:


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/kv-complaints
!ls results/manifest.json data/review_texts.json

In [ ]:
!pip install -q "kvpress==0.5.5" "transformers==4.57.*" accelerate pytest
!pip freeze > requirements.lock
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "fire>=0.7"

## Git
`run.py` records the commit of every run, and Test 4 checks every committed file.

In [ ]:
!git config --global --add safe.directory '*'
!git config --global user.name "PranavaMR"
!git config --global user.email "aragornoftirith@gmail.com"
!test -d .git || git init -q
!git add -A && git commit -qm "Freeze code, lexicon and templates (code drafted with Claude)" || true
!git log --oneline | head -3

## Smoke test: KnormPress at 50% on a ~2k-token dummy context

In [ ]:

import sys, shlex
sys.modules['pipes'] = shlex

import torch, kvpress
from kvpress import KnormPress
from run import load_pipe

pipe = load_pipe()
dummy = ' '.join(['The soup was warm and the staff were kind.'] * 200)
print(pipe(dummy, question='\n\nWhat did people say about the soup?',
           press=KnormPress(compression_ratio=0.5), max_new_tokens=40)['answer'])
print('model revision:', pipe.model.config._commit_hash)
del pipe; torch.cuda.empty_cache()

## Tests 1 to 4
`-rs` lists skipped tests: all four must say passed, none skipped.

In [ ]:
!python -m pytest -q -rs tests

## Calibration
Full cache, 12 planted contexts, Q1 only. If Full mentions the complaint in under 30% or over 95%,
change `N_RECOMMENDATIONS` in `config.py` **once** (to 8 or 3), log it in `logs/decisions.md`, commit, then launch.

In [ ]:
!python run.py --calibrate

In [ ]:
import json
from transformers import AutoTokenizer
import config
from detect import keyword_mention, citation_mention, well_formed

tok = AutoTokenizer.from_pretrained(config.MODEL_NAME)
for line in open(config.CALIBRATION_FILE):
    r = json.loads(line)
    a = r["q1_answer"]
    print(f"{r['context_id']}  {r['plant_aspect']:13}  tokens={len(tok.encode(a)):3}  "
          f"kw={keyword_mention(a, r['plant_aspect'])!s:5}  cite={citation_mention(a, r['plant_position'])!s:5}  "
          f"wf={well_formed(a)!s:5}  ends: ...{a[-40:]!r}")

In [ ]:
!cp data/calibration_raw.jsonl data/calibration_256tok.jsonl

## The 240 runs
Roughly 1 to 1.5 h on a T4.

In [ ]:
!python run.py

In [ ]:
!wc -l data/runs_raw.jsonl data/calibration_raw.jsonl

## Redact, label, analyse
1. `redact.py` writes the committable `results/runs.jsonl`.
2. `labels.py` writes `labels/to_label.csv`


In [ ]:
!python redact.py
!python labels.py

In [ ]:
!python analysis.py

In [ ]:
from IPython.display import Image, display
display(Image('results/figures/fig1_mention_rate.png'))
display(Image('results/figures/fig2_retention.png'))

## Final commit


In [ ]:
!git config user.name "PranavaMR"
!git config user.email "aragornoftirith@gmail.com"
!git add -A
!git commit -qm "Runs, labels, tables, figures, memo and decisions log" && git tag v-assessment
!git log --oneline --decorate | head -3

In [ ]:
!printf '\n# Google Drive shortcut files\n*.gsheet\n*.gdoc\n*.gslides\n' >> .gitignore
!git add -A
!git status --short | grep "data/" || echo "OK: no data/ files staged"
!python -m pytest -q tests/test_licence.py
!git commit -q --amend --no-edit
!git tag -f v-assessment
!git show --stat --oneline HEAD | tail -20

**Analysis of wrong outputs**

In [ ]:
import json, config
from collections import Counter
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(config.MODEL_NAME)
hit, total = Counter(), Counter()
for line in open(config.RAW_RUNS_FILE):
    r = json.loads(line)
    total[r["press"]] += 1
    hit[r["press"]] += len(tok.encode(r["q1_answer"])) >= config.Q1_MAX_NEW_TOKENS - 2
for m in total: print(f"{m:14} {hit[m]}/{total[m]} hit the {config.Q1_MAX_NEW_TOKENS}-token limit")

In [ ]:
import json, config
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(config.MODEL_NAME)
loops = cut = 0
for line in open(config.RAW_RUNS_FILE):
    r = json.loads(line)
    a = r["q1_answer"]
    if r["press"] != "full" or len(tok.encode(a)) < config.Q1_MAX_NEW_TOKENS - 2:
        continue
    if a[-300:].count("Review") >= 6:
        loops += 1                      # stuck listing reviews
    else:
        cut += 1                        # still writing real advice
        print(r["context_id"], "...", a[-120:].replace("\n", " "))
print(f"{loops} loops, {cut} cut off mid-advice")

In [ ]:
!python labels.py